In [1]:
import pandas as pd

In [2]:
pd.set_option('display.max_columns', None)  
pd.set_option('display.max_rows', None)  

In [ ]:
# read dfVulCWE (BigVul Commits processed)

In [ ]:
# all path need to be modified accordingly

In [3]:
dfVulCWE = pd.read_csv("dfVulCWE.csv")

In [9]:
import pandas as pd

# Group by commit_id and check if there are multiple distinct CWE IDs
cwe_per_commit = dfVulCWE.groupby('commit_id')['CVE ID'].nunique()

# Filter the commits where there are more than one distinct CWE IDs
commits_with_multiple_cwes = cwe_per_commit[cwe_per_commit > 1]

# Display the results
print("Commits with multiple distinct CWE IDs:")
print(commits_with_multiple_cwes)


Commits with multiple distinct CWE IDs:
Series([], Name: CVE ID, dtype: int64)


In [ ]:
# new code to get comit urls

In [4]:
import pandas as pd

# First, select the necessary columns and drop duplicates based on 'codeLink'
df_filtered = dfVulCWE[['project', 'codeLink', 'commit_id', 'CWE ID', 'CVE ID']].drop_duplicates()

# Filter rows where 'codeLink' starts with 'https://github.com/'
df_filtered = df_filtered[df_filtered['codeLink'].str.startswith('https://github.com/', na=False)]

# Order by project
df_filtered = df_filtered.sort_values(by='project')

# Save the filtered DataFrame to a CSV
df_filtered.to_csv("commitUrlFinalBigVul.csv", index=False)

# Create a new DataFrame with only the distinct commit URLs (codeLink)
dfResult = df_filtered[['codeLink']].drop_duplicates()

# Convert the 'codeLink' column to a list of commit URLs
commit_urls = dfResult['codeLink'].tolist()

# Optionally, display the commit URLs
#print(commit_urls)


In [ ]:
# get file extensions to check which one to discard

In [ ]:
# Github Token need to be added

In [19]:
import requests
import os
import shutil
from urllib.parse import urlparse
import re
from base64 import b64decode
from datetime import datetime
import difflib
import time
import logging
import csv

#list of GitHub personal access tokens
GITHUB_TOKENS = [

]

# Configure logging
LOG_FILE_PATH = r"M:\FULL_DATA_COLLECTED\commit_errors_log.txt"
logging.basicConfig(
    filename=LOG_FILE_PATH,
    level=logging.ERROR,
    format='[%(asctime)s] %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

# Global index to track the current token being used
current_token_index = 0
# Timeout setting for requests
REQUEST_TIMEOUT = 10  # seconds
# Retry settings
MAX_RETRIES = 3
RETRY_DELAY = 5  # seconds
# Delay settings
FETCH_DELAY = 0.5  # seconds between each fetch
FILE_PROCESS_DELAY = 0.5  # seconds between processing each file
COMMIT_PROCESS_DELAY = 2  # seconds between processing each commit

# Path to the execution time log CSV file
TIME_LOG_PATH = r"M:\FULL_DATA_COLLECTED\execution_time_log.csv"

# Global set to track logged errors
logged_errors = set()

# Function to get the current token
def get_current_token():
    global current_token_index
    return GITHUB_TOKENS[current_token_index]

# Function to switch to the next token
def switch_to_next_token():
    global current_token_index
    current_token_index = (current_token_index + 1) % len(GITHUB_TOKENS)

# Function to check GitHub API rate limit
def check_rate_limit():
    current_token = get_current_token()
    headers = {'Authorization': f'token {current_token}'}
    rate_limit_url = "https://api.github.com/rate_limit"
    response = requests.get(rate_limit_url, headers=headers, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    rate_limit_data = response.json()

    remaining = rate_limit_data['rate']['remaining']

    # Automatically switch token if remaining requests are low
    if remaining < 1000:
        switch_to_next_token()

# General function to log errors with description and index
def log_error(commit_url, error_message, index=None):
    error_key = (commit_url, error_message)
    if error_key not in logged_errors:
        log_message = f"Commit URL: {commit_url}\n  - Error: {error_message}"
        if index is not None:
            log_message = f"[Index: {index}] {log_message}"
        logging.error(log_message)
        logged_errors.add(error_key)  # Add to logged errors to avoid duplicate logging

# Function to perform a GET request with retries, including 404 error handling
def get_with_retries(url, headers, params=None, index=None):
    for attempt in range(MAX_RETRIES):
        try:
            response = requests.get(url, headers=headers, params=params, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()
            # Delay between fetches
            time.sleep(FETCH_DELAY)
            return response
        except requests.exceptions.HTTPError as e:
            if response.status_code == 404:
                error_message = f"404 Not Found - The requested resource could not be found."
                log_error(url, error_message, index)  # Pass index
                return None  # Return None to indicate the 404 error
            else:
                error_message = f"HTTPError - {str(e)}"
                log_error(url, error_message, index)  # Pass index
                time.sleep(RETRY_DELAY)
        except (requests.exceptions.ConnectTimeout, requests.exceptions.ReadTimeout) as e:
            error_message = f"ConnectionTimeout - {str(e)}"
            log_error(url, error_message, index)  # Pass index
            time.sleep(RETRY_DELAY)
        except requests.exceptions.RequestException as e:
            error_message = f"Request failed: {str(e)}"
            log_error(url, error_message, index)  # Pass index
            break
    raise Exception(f"Failed to get a response from {url} after {MAX_RETRIES} attempts.")

# Function to fetch all commits for a file, handling pagination
def fetch_all_commits_for_file(repo_owner, repo_name, file_path, index):
    commits_url = f'https://api.github.com/repos/{repo_owner}/{repo_name}/commits'
    params = {'path': file_path, 'per_page': 100}  # Adjust per_page as needed
    all_commits = []
    page = 1

    while True:
        headers = {'Authorization': f'token {get_current_token()}'}
        params['page'] = page
        response = get_with_retries(commits_url, headers, params=params, index=index)
        if response is None:
            log_error(commits_url, "404 error when fetching commits", index)
            break
        commits = response.json()
        if not commits:
            break
        all_commits.extend(commits)
        page += 1

    return all_commits

# Function to check if a directory is empty and remove it if it is
def remove_empty_directory(directory):
    if os.path.exists(directory) and not os.listdir(directory):  # Check if the directory exists and is empty
        shutil.rmtree(directory)  # Remove the directory and its contents
        print(f"Removed empty directory: {directory}")

# List of known non-code file extensions
NON_CODE_EXTENSIONS = ['', '.txt', '.md', '.jpg', '.png', '.jpeg', '.pdf', '.xml', '.conf', '.man', '.texi']

# Function to check if a file is non-code
def is_non_code(file_name):
    file_extension = os.path.splitext(file_name)[1]
    return file_extension in NON_CODE_EXTENSIONS

# Function to log execution time to a CSV file
def log_execution_time(range_str, execution_time):
    file_exists = os.path.isfile(TIME_LOG_PATH)

    # Open the file in append mode
    with open(TIME_LOG_PATH, mode='a', newline='') as file:
        writer = csv.writer(file)
        # Write the header if the file doesn't exist
        if not file_exists:
            writer.writerow(["Range", "Total Execution Time (seconds)"])
        # Append the new row with the range and execution time
        writer.writerow([range_str, execution_time])

# Function to process commit links
def process_commit_link(commit_url, index):
    try:
        # Parse the URL to extract repository details and commit hash
        parsed_url = urlparse(commit_url)
        path = parsed_url.path

        # Extract repo owner, repo name, and commit hash using regex
        match = re.match(r'/([^/]+)/([^/]+)/commit/([a-f0-9]+)', path)
        if match:
            REPO_OWNER = match.group(1)
            REPO_NAME = match.group(2)
            COMMIT_HASH = match.group(3)
        else:
            error_message = f"Invalid commit URL: {commit_url}"
            log_error(commit_url, error_message, index)
            return

        # Base directories
        FULL_DATA_DIR = r"M:\FULL_DATA_COLLECTED"
        PROJECT_DIR = os.path.join(FULL_DATA_DIR, REPO_NAME)  # Project name folder
        COMMIT_DIR = os.path.join(PROJECT_DIR, f"{REPO_NAME}_{COMMIT_HASH}")  # Folder named projectname_commithash
        OUTPUT_DIR = os.path.join(COMMIT_DIR, 'all_versions')
        COMMIT_MESSAGES_DIR = os.path.join(COMMIT_DIR, 'commit_messages')
        CHANGES_DIR = os.path.join(COMMIT_DIR, 'file_changes_in_versions')

        # Check and display API rate limit before processing
        check_rate_limit()

        # GitHub API endpoint for the commit
        commit_api_url = f'https://api.github.com/repos/{REPO_OWNER}/{REPO_NAME}/commits/{COMMIT_HASH}'

        # Fetch the commit details
        headers = {'Authorization': f'token {get_current_token()}'}
        response = get_with_retries(commit_api_url, headers, index=index)
        if response is None:
            return
        commit_data = response.json()

        # Get the list of files changed in the commit
        files = commit_data['files']

        if files:  # Proceed only if there are files to process
            # Fetch the complete commit history for each file
            for file_info in files:
                file_path = file_info['filename']

                # Filter out non-code files
                if is_non_code(file_path):
                    continue

                # Proceed with the rest of your processing only for source code files
                commits = fetch_all_commits_for_file(REPO_OWNER, REPO_NAME, file_path, index)

                # Find the commit before the `fixed_version` (i.e., the commit just before COMMIT_HASH)
                commits.reverse()  # Reverse the order to process in chronological order
                fixed_version_index = None
                for idx, commit in enumerate(commits):
                    if commit['sha'] == COMMIT_HASH:
                        fixed_version_index = idx
                        break

                if fixed_version_index is None:
                    error_message = f"Fixed version {COMMIT_HASH} not found in commit history for {file_path}."
                    log_error(commit_url, error_message, index)  # Log error if fixed version is not found
                    continue

                # The commit before the fixed version (if it exists)
                if fixed_version_index > 0:
                    commits_to_process = commits[fixed_version_index - 1:]  # The previous commit and all subsequent commits
                else:
                    commits_to_process = commits[fixed_version_index:]  # No commit before, process from fixed version onward

                # Create output directories if they don't exist
                os.makedirs(OUTPUT_DIR, exist_ok=True)
                os.makedirs(COMMIT_MESSAGES_DIR, exist_ok=True)
                os.makedirs(CHANGES_DIR, exist_ok=True)

                previous_content = None
                version_number = 1

                # Process each commit in chronological order starting from the commit before the `fixed_version`
                for commit in commits_to_process:
                    sha = commit['sha']
                    commit_date = commit['commit']['committer']['date']
                    formatted_date = datetime.strptime(commit_date, '%Y-%m-%dT%H:%M:%SZ').strftime('%Y%m%d_%H%M%S')
                    commit_message = commit['commit']['message'].strip().replace('\n', ' ')  # Clean commit message
                    file_url = f'https://api.github.com/repos/{REPO_OWNER}/{REPO_NAME}/contents/{file_path}?ref={sha}'

                    # Check if this is the specific commit we're working on (fixed version)
                    is_fixed_version = (sha == COMMIT_HASH)
                    version_suffix = "_fixed_version" if is_fixed_version else ""

                    # Try to fetch the file content
                    try:
                        headers = {'Authorization': f'token {get_current_token()}'}
                        file_response = get_with_retries(file_url, headers, index=index)
                        if file_response is None:
                            continue  # Skip to the next file if 404 error occurred
                        file_content_base64 = file_response.json()['content']
                        file_content = b64decode(file_content_base64).decode('utf-8')

                        # Save the file content in the 'all_versions' directory
                        file_name = os.path.basename(file_path)
                        file_version_path = os.path.join(OUTPUT_DIR, f"{file_name}_v{version_number}_{formatted_date}_{sha}{version_suffix}.txt")
                        with open(file_version_path, 'w', encoding='utf-8') as f:
                            f.write(file_content)

                        # Save the commit message in the 'commit_messages' directory
                        commit_message_path = os.path.join(COMMIT_MESSAGES_DIR, f"{file_name}_v{version_number}_{formatted_date}_{sha}_commit_message{version_suffix}.txt")
                        with open(commit_message_path, 'w', encoding='utf-8') as f:
                            f.write(commit_message)

                        # If there's a previous version, compare and save the changes
                        if previous_content is not None:
                            diff = difflib.unified_diff(previous_content.splitlines(), file_content.splitlines(),
                                                        fromfile=f'v{version_number-1}', tofile=f'v{version_number}')
                            changes_file_path = os.path.join(CHANGES_DIR, f"{file_name}_changes_v{version_number-1}_v{version_number}{version_suffix}.txt")
                            with open(changes_file_path, 'w', encoding='utf-8') as f:
                                f.write('\n'.join(diff))

                        # Update previous content for the next iteration
                        previous_content = file_content
                        version_number += 1  # Increment version number

                        # Add delay after processing each file version
                        time.sleep(FILE_PROCESS_DELAY)

                    except Exception as e:
                        error_message = f"Unexpected error while processing the file: '{file_path}'. Error: {str(e)}"
                        log_error(commit_url, error_message, index)

        # Check if the commit directory is empty and remove it if no files were saved
        remove_empty_directory(COMMIT_DIR)

        # Check and display API rate limit after processing
        check_rate_limit()

    except Exception as e:
        error_message = f"Exception - {str(e)}"
        log_error(commit_url, error_message, index)

# Example usage for processing commit URLs
start_time = time.time()  # Start the timer before processing the first commit in the range

# Define the range for processing commits (e.g., 800-803)
start_index = 401
end_index = 450

for index in range(start_index, end_index + 1):
    try:
        process_commit_link(commit_urls[index], index)
        print(f"{index}")
    except Exception as e:
        log_error(commit_urls[index], f"Error processing commit at index {index}. Error: {str(e)}", index)
    time.sleep(COMMIT_PROCESS_DELAY)

# Calculate the total execution time after all commits in the range are processed
end_time = time.time()
total_execution_time = end_time - start_time  # Calculate total execution time for the whole range

# Log the execution time for the provided range
log_execution_time(f"{start_index}-{end_index}", total_execution_time)

print("done")


401
402
403
404
405
406
407
408
409
410
411
412
413
414
415
416
417
418
419
420
421
422
423
424
425
426
427
428
429
430
431
432
433
434
435
436
437
438
439
440
441
442
443
444
445
446
447
448
449
450
done
